# Training

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = 'https://github.com/Tienkhoaa2908/ISAS2026-DLCS.git'
REPO_NAME = 'ISAS2026-DLCS'

if Path('/content').exists():
    work = Path('/content') / REPO_NAME
    if not work.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(work)], check=True)
else:
    cwd = Path.cwd()
    work = cwd if (cwd / 'requirements.txt').exists() else cwd.parent

work = work.resolve()
private_root = Path(os.environ['ISAS2026_PRIVATE_DIR']).expanduser().resolve() if os.environ.get('ISAS2026_PRIVATE_DIR') else (
    Path('/content/ISAS2026-private') if Path('/content').exists() else work.parent / f'{work.name}-private'
)
private_root = private_root.resolve()
if private_root == work or work in private_root.parents:
    raise ValueError('ISAS2026_PRIVATE_DIR must be outside the repository worktree.')
private_root.mkdir(parents=True, exist_ok=True)
cache_root = private_root / 'cache'
for name in ('pip', 'matplotlib', 'joblib'):
    (cache_root / name).mkdir(parents=True, exist_ok=True)
os.environ['PIP_CACHE_DIR'] = str(cache_root / 'pip')
os.environ['MPLCONFIGDIR'] = str(cache_root / 'matplotlib')
os.environ['JOBLIB_TEMP_FOLDER'] = str(cache_root / 'joblib')
os.chdir(work)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repository =', Path.cwd())


## Data construction

In [ ]:
from pathlib import Path
import os, subprocess, sys

archive = Path(os.environ.get('DATA_ARCHIVE', str(private_root / 'data.zip'))).expanduser().resolve()
manifest = Path(os.environ.get('DATA_MANIFEST', str(private_root / 'data_manifest.json'))).expanduser().resolve()
for name, path in (('DATA_ARCHIVE', archive), ('DATA_MANIFEST', manifest)):
    if path == work or work in path.parents:
        raise ValueError(f'{name} must point outside the repository worktree.')

if not archive.is_file() or not manifest.is_file():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Set DATA_ARCHIVE and DATA_MANIFEST to input files outside the repository worktree.') from exc
    upload_cwd = Path.cwd()
    try:
        os.chdir(private_root)
        uploaded = files.upload()
    finally:
        os.chdir(upload_cwd)
    try:
        if not archive.is_file():
            archive = private_root / Path(next(name for name in uploaded if Path(name).suffix.lower() == '.zip')).name
        if not manifest.is_file():
            manifest = private_root / Path(next(name for name in uploaded if Path(name).suffix.lower() == '.json')).name
    except StopIteration as exc:
        raise FileNotFoundError('Upload both the input ZIP archive and manifest, or set DATA_ARCHIVE and DATA_MANIFEST.') from exc

run_root = private_root / 'runtime'
runtime = run_root / 'Dataset'
run_root.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, 'scripts/prepare_data.py',
    '--archive', str(archive), '--manifest', str(manifest), '--out', str(runtime)
], check=True)
print('data_root =', runtime)


In [ ]:
import subprocess, sys, pandas as pd

subprocess.run([
    sys.executable, 'scripts/verify_fingerprints.py',
    '--data-root', str(runtime),
    '--out', str(run_root / 'rebuilt_train_fingerprints.csv')
] , check=True)

fp = pd.read_csv(runtime / 'train_fingerprints.csv')
feature_cols = [c for c in fp.columns if c not in ['date', 'win_start', 'win_end', 'room']]
print('rows =', len(fp))
print('classes =', fp['room'].astype(str).nunique())
print('numeric_features =', len(feature_cols))
display(fp['room'].astype(str).value_counts().rename('support').to_frame())


## Repeated session-grouped validation

In [ ]:
import subprocess, sys, pandas as pd

subprocess.run([
    sys.executable, 'scripts/session_grouped_validation.py',
    '--data-root', str(runtime),
    '--outdir', str(run_root / 'session_grouped'),
    '--trees', '1200'
] , check=True)

summary = pd.read_csv(run_root / 'session_grouped' / 'confirmation_summary.csv')
by_rep = pd.read_csv(run_root / 'session_grouped' / 'confirmation_by_rep.csv')
display(summary)
display(by_rep)


## Leave-one-day-out validation

In [ ]:
import subprocess, sys, pandas as pd

subprocess.run([
    sys.executable, 'scripts/lodo_validation.py',
    '--data-root', str(runtime),
    '--outdir', str(run_root / 'lodo'),
    '--specialist-trees', '1200',
    '--force'
] , check=True)

agg = pd.read_csv(run_root / 'lodo' / 'paper_lodo_aggregate.csv')
by_day = pd.read_csv(run_root / 'lodo' / 'paper_lodo_by_day.csv')
display(agg)
display(by_day.pivot(index='held_out_day', columns='system', values='macro_f1'))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

pred = pd.read_csv(run_root / 'lodo' / 'paper_lodo_predictions.csv')
classes = sorted(pred['true'].astype(str).unique())
system = 'main_centered'
y_true = pred['true'].astype(str)
y_pred = pred[system].astype(str)

report = (pd.DataFrame(classification_report(
    y_true, y_pred, labels=classes, target_names=classes,
    output_dict=True, zero_division=0
)).T.loc[classes, ['precision', 'recall', 'f1-score', 'support']])
display(report)

cm = confusion_matrix(y_true, y_pred, labels=classes, normalize='true')
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(classes)), classes, rotation=90)
ax.set_yticks(range(len(classes)), classes)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title('Fixed-22-class LODO — main + centered smoothing')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 4))
plt.bar(report.index, report['f1-score'])
plt.xticks(rotation=90)
plt.ylim(0, 1)
plt.ylabel('F1')
plt.xlabel('Location class')
plt.tight_layout()
plt.show()
